# Лекция: Дисперсионный анализ (ANOVA / MANOVA) в Python

**Дисциплина:** Введение в анализ больших данных

**ANOVA** проверяет H0: средние отклика одинаковы по уровням фактора(ов).

- однофакторный: одна категориальная независимая;
- двухфакторный: два фактора + взаимодействие (`y ~ A * B`);
- **MANOVA**: несколько зависимых переменных сразу.

Инструменты: `statsmodels` (`ols`, `anova_lm`, `MANOVA`).

Демо: **tips** и **penguins**. Примеры **не из лабораторного задания** — его выполните самостоятельно.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.anova import anova_lm
from statsmodels.multivariate.manova import MANOVA

plt.rcParams["figure.figsize"] = (10, 6)
sns.set_style("whitegrid")
print("Библиотеки загружены")


---
## 1. Двухфакторный ANOVA

Отклик `tip`, факторы `day` и `time` (датасет tips).

Формула `tip ~ day * time` = главные эффекты + взаимодействие.


In [ ]:
tips = sns.load_dataset("tips")
print(tips.head())
print("\nЧастоты day × time:")
print(pd.crosstab(tips["day"], tips["time"]))


In [ ]:
agg = tips.groupby(["day", "time"])["tip"].agg(["mean", "std", "count"]).round(2)
print(agg)


In [ ]:
tips = tips.copy()
tips["day"] = tips["day"].astype("category")
tips["time"] = tips["time"].astype("category")

model = smf.ols("tip ~ day * time", data=tips).fit()
print(anova_lm(model, typ=2))


**Чтение таблицы ANOVA**

- для каждой строки: Sum Sq, F, PR(>F);
- p < 0.05 → эффект **значим**;
- значимое **взаимодействие** `day:time` значит, что влияние дня зависит от времени (обед/ужин).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.boxplot(data=tips, x="day", y="tip", hue="time", ax=axes[0])
axes[0].set_title("Boxplot: tip по day и time")
sns.pointplot(data=tips, x="day", y="tip", hue="time", errorbar="se", ax=axes[1])
axes[1].set_title("Interaction plot (средние ± SE)")
plt.tight_layout()
plt.show()


---
## 2. Однофакторный ANOVA

Пример: зависит ли `total_bill` от `day`?


In [ ]:
m1 = smf.ols("total_bill ~ day", data=tips).fit()
print(anova_lm(m1, typ=2))
print(tips.groupby("day")["total_bill"].mean().round(2))


---
## 3. MANOVA

Несколько зависимых переменных одновременно.  
H0: векторы средних одинаковы по группам.


In [ ]:
peng = sns.load_dataset("penguins").dropna()
ys = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm"]
print("Средние по species:")
print(peng.groupby("species")[ys].mean().round(2))


In [ ]:
formula = "bill_length_mm + bill_depth_mm + flipper_length_mm ~ C(species)"
maov = MANOVA.from_formula(formula, data=peng)
print(maov.mv_test())


После значимого MANOVA смотрят **одномерные** ANOVA по каждой y.


In [ ]:
print("=== Одномерные ANOVA ===")
for y in ys:
    m = smf.ols(f"{y} ~ C(species)", data=peng).fit()
    print(f"\n{y}:")
    print(anova_lm(m, typ=2))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y in zip(axes, ys):
    sns.boxplot(data=peng, x="species", y=y, ax=ax)
    ax.set_title(y)
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.show()


### Как описать результаты

1. **ANOVA:** для A, B, A:B — F и p; p < 0.05 → эффект значим.  
2. Значимое взаимодействие — влияние одного фактора меняется по уровням другого.  
3. **MANOVA:** p < 0.05 → группы различаются в пространстве нескольких откликов.  
4. Затем — одномерные ANOVA и графики (boxplot / pointplot).

---
## Шпаргалка по методам (Python)

| Задача | Код |
|--------|-----|
| ANOVA-модель | `smf.ols("y ~ A * B", data=df).fit()` |
| Таблица ANOVA | `anova_lm(model, typ=2)` |
| Средние по группам | `df.groupby(["A","B"])["y"].mean()` |
| MANOVA | `MANOVA.from_formula("y1 + y2 ~ F", data=df)` |
| Результат MANOVA | `maov.mv_test()` |
| Interaction plot | `sns.pointplot(..., hue=...)` |

---
## Что сделать после лекции

1. Повторите `ols` + `anova_lm` и MANOVA на **других** столбцах.  
2. Откройте лабораторное задание и выполните ANOVA/MANOVA **самостоятельно** на своих CSV.  
3. Факторы удобно кодировать как `category`.

Удачи!
